# Kit de Reprodução — Math Benchmark (PT-BR / PT-PT) — Colab Edition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deep-spin/math-benchmark/blob/main/kit-reproducao-colab.ipynb)

Este notebook reúne **todo o conteúdo do projeto** [`deep-spin/math-benchmark`](https://github.com/deep-spin/math-benchmark) — que originalmente estava espalhado em vários notebooks (`create_prompts.ipynb`, `create_judge_prompts.ipynb`, `evaluate-multiple-choice-answers.ipynb`, `evaluate-multiple-choice-answers-variability.ipynb`, `evaluate-open-ended-answers.ipynb`) e no script `assync_send_prompts.py` — em **um único notebook**, pronto para rodar do início ao fim no Google Colab, sem depender do ambiente local.

## O que este notebook faz

1. **Setup**: instala dependências e clona o repositório com todos os dados (`data/`), prompts (`prompts/`, `judge-prompts/`) e resultados (`results/`, `judge-results/`) já publicados.
2. **Reproduz os resultados já publicados** (Parte 3) — calcula os relatórios de acurácia (múltipla escolha, variabilidade entre execuções, e questões abertas via juiz) a partir dos dados já existentes no repositório. **Não requer chave de API.**
3. **Compara com o artigo (MATH-PT)** (Parte 4) — confronta, tabela por tabela, os números publicados no artigo com os recalculados neste notebook, com interpretação didática das diferenças (ou da falta delas). **Não requer chave de API.**
4. **Gera novos dados do zero** (Parte 5, opcional) — cria prompts a partir de `data/*.json`, envia para um modelo (via OpenRouter ou um servidor vLLM local) e avalia as respostas. **Requer uma chave de API** (`OPEN_ROUTER_API_KEY`) ou um servidor local compatível com a API da OpenAI.
5. **Exporta os resultados** (Parte 6) — compacta a pasta `results/` para download ou salva no Google Drive.

## Estrutura

| Parte | Conteúdo | Equivalente ao projeto original |
|---|---|---|
| 0 | Setup do ambiente | — |
| 1 | Credenciais de API | `.env` |
| 2 | Núcleo assíncrono de envio de prompts + funções de avaliação | `assync_send_prompts.py` |
| 3 | Reprodução dos resultados publicados | `evaluate-multiple-choice-answers*.ipynb`, `evaluate-open-ended-answers.ipynb` |
| 4 | Comparação com o artigo (MATH-PT) + interpretação didática | — |
| 5 | Geração de novos dados (opcional) | `create_prompts.ipynb`, `create_judge_prompts.ipynb` |
| 6 | Exportação dos resultados | — |

> Rode as células em ordem. As Partes 0–2 são pré-requisito para todas as demais.

## Parte 0 — Setup do ambiente


In [ ]:
!pip install -q openai tenacity tqdm python-dotenv nest_asyncio pandas matplotlib
print("Dependências instaladas.")

In [ ]:
import os

REPO_URL = "https://github.com/deep-spin/math-benchmark.git"
REPO_DIR = "/content/math-benchmark"

if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print("Repositório já clonado, atualizando...")
    !git -C {REPO_DIR} pull -q

os.chdir(REPO_DIR)
print("Diretório de trabalho:", os.getcwd())
print(sorted(os.listdir(".")))


## Parte 1 — Credenciais de API

Necessário apenas se você for **gerar novos dados** (Parte 5). Para apenas **reproduzir os resultados já publicados** (Parte 3), pule esta célula.

A chave é lida do ambiente/`.env` como `OPEN_ROUTER_API_KEY` (mesma variável usada pelo projeto original).

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPEN_ROUTER_API_KEY"):
    key = getpass("Cole sua OPEN_ROUTER_API_KEY (Enter para pular): ")
    if key:
        os.environ["OPEN_ROUTER_API_KEY"] = key
        with open(".env", "w", encoding="utf-8") as f:
            f.write(f"OPEN_ROUTER_API_KEY={key}\n")
        print("Chave configurada.")
    else:
        print("Nenhuma chave fornecida. Você ainda pode rodar a Parte 3 (reprodução de resultados existentes).")
else:
    print("OPEN_ROUTER_API_KEY já definida no ambiente.")


**(Opcional) Persistir resultados no Google Drive.** Descomente e rode se quiser que os arquivos gerados sobrevivam ao encerramento da sessão do Colab.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')


## Parte 2 — Núcleo assíncrono de envio de prompts

Equivalente Colab-friendly de `assync_send_prompts.py`: mesma lógica de retry, escrita em JSONL com *resume* (pula ids já processados) e limite de concorrência via semáforo — mas exposta como uma função `async` (`send_prompts_to_model`) chamável diretamente de uma célula com `await`, em vez de um script de linha de comando.

`nest_asyncio` é aplicado para permitir `await` em qualquer célula do notebook.


In [ ]:
import asyncio
import json
import os
import re
from typing import Any, Dict, Iterable, Set

import tenacity
from openai import APITimeoutError, AsyncOpenAI
from tqdm.auto import tqdm
from dotenv import load_dotenv

import nest_asyncio
nest_asyncio.apply()

load_dotenv()


def iter_jsonl(path: str) -> Iterable[Dict[str, Any]]:
    """Yield JSON objects from a JSONL file, skipping invalid/blank lines."""
    if not os.path.exists(path):
        return
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception:
                continue


def load_processed_ids(path: str) -> Set[Any]:
    """Collect the set of already-processed ids from an existing JSONL file."""
    ids: Set[Any] = set()
    for obj in iter_jsonl(path) or []:
        qid = obj.get("id") if isinstance(obj, dict) else None
        if qid is not None:
            ids.add(qid)
    return ids


def append_jsonl(path: str, obj: Dict[str, Any]) -> None:
    """Append a single JSON object as one line to a JSONL file (fsync for durability)."""
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    data = (json.dumps(obj, ensure_ascii=False) + "\n").encode("utf-8")
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_APPEND, 0o644)
    try:
        os.write(fd, data)
        os.fsync(fd)
    finally:
        os.close(fd)


class ChatModel:
    def __init__(self, model_name, base_url, api_key, timeout=1800):
        self.client = AsyncOpenAI(base_url=base_url, api_key=api_key, timeout=timeout)
        self.model_name = model_name

    @tenacity.retry(
        wait=tenacity.wait_exponential(multiplier=1, min=4, max=10) + tenacity.wait_random(0, 2),
        retry=tenacity.retry_if_exception(lambda exc: not isinstance(exc, APITimeoutError)),
        stop=tenacity.stop_after_attempt(3),
        reraise=True,
    )
    async def __call__(self, question_text: str) -> dict:
        completion = await self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": question_text}],
        )
        choice = completion.choices[0]
        message = choice.message
        content = getattr(message, "content", None)
        finish_reason = getattr(choice, "finish_reason", None)

        # Do not retry if the model stopped due to a length/token limit.
        if finish_reason == "length":
            return {
                "choice.message.content": content,
                "finish_reason": finish_reason,
                "model": getattr(completion, "model", None),
                "usage": completion.usage.model_dump() if getattr(completion, "usage", None) else None,
            }

        if isinstance(content, str):
            if content.strip():
                return {
                    "choice.message.content": content,
                    "finish_reason": finish_reason,
                    "model": getattr(completion, "model", None),
                    "usage": completion.usage.model_dump() if getattr(completion, "usage", None) else None,
                }
            raise ValueError("Received empty or whitespace-only content from API")
        elif content is not None:
            return {
                "choice.message.content": content,
                "finish_reason": finish_reason,
                "model": getattr(completion, "model", None),
                "usage": completion.usage.model_dump() if getattr(completion, "usage", None) else None,
            }
        else:
            raise ValueError("Received null content from API")


async def send_prompts_to_model(
    prompts_path: str,
    responses_path: str,
    model: str,
    base_url: str,
    api_key: str,
    concurrency: int = 8,
):
    """Colab-friendly equivalent of assync_send_prompts.py. Resumable: skips ids already present in responses_path."""
    if api_key == "openrouter":
        api_key = os.getenv("OPEN_ROUTER_API_KEY")
        if not api_key:
            raise RuntimeError("OPEN_ROUTER_API_KEY não definida. Rode a célula de credenciais (Parte 1).")

    chat = ChatModel(model_name=model, base_url=base_url, api_key=api_key)

    with open(prompts_path, "r", encoding="utf-8") as f:
        prompts = json.load(f)

    processed_ids = load_processed_ids(responses_path)
    pending = [e for e in prompts if e.get("id") not in processed_ids]

    sem = asyncio.Semaphore(max(1, concurrency))
    write_lock = asyncio.Lock()

    async def process(entry: Dict[str, Any]):
        async with sem:
            qid = entry.get("id")
            prompt_text = entry["prompt"]
            try:
                raw_response = await chat(prompt_text)
                record = {"id": qid, "raw_response": raw_response}
            except APITimeoutError as e:
                record = {"id": qid, "error": f"APITimeoutError: {str(e)}"}
            except Exception as e:
                record = {"id": qid, "error": f"Exception: {str(e)}"}
            async with write_lock:
                append_jsonl(responses_path, record)

    tasks = [asyncio.create_task(process(e)) for e in pending]
    with tqdm(total=len(pending), desc=f"Enviando prompts ({model})", unit="prompt") as pbar:
        for fut in asyncio.as_completed(tasks):
            try:
                await fut
            finally:
                pbar.update(1)

    print(f"Concluído: {len(pending)} novos, {len(processed_ids)} já existentes em {responses_path}")


print("Núcleo assíncrono pronto — use `await send_prompts_to_model(...)` em qualquer célula.")


### Funções de avaliação reutilizáveis

Extraem a resposta final (`\boxed{...}`) das respostas de múltipla escolha e calculam relatórios de acurácia por nível/presença de figura — mesma lógica de `evaluate-multiple-choice-answers*.ipynb` e `evaluate-open-ended-answers.ipynb`, encapsulada em funções reaproveitadas tanto na reprodução (Parte 3) quanto na comparação com o artigo (Parte 4) e na geração de novos dados (Parte 5).

In [ ]:
score_re = re.compile(r'"score"\s*:\s*(0|1)')
explanation_re = re.compile(r'"explanation"\s*:\s*"([^"]*)"')


def parse_boxed_letter(text):
    """Extract the final boxed letter answer (\\boxed{X}) from a model response."""
    if not text:
        return None
    m = re.search(r'(?<=\\boxed\{)([A-Za-z])(?=\}(?!.*\\boxed))', text)
    return m.group(0) if m else None


def _extract_content(response_record):
    raw = response_record.get("raw_response")
    return raw.get("choice.message.content") if isinstance(raw, dict) else None


def compute_mc_accuracy(models, responses_by_model, prompts_by_id, level_range=range(1, 6)):
    """Compute multiple-choice accuracy stats (overall / with-figure / without-figure) per model and level."""
    def init_stats():
        return {m: {lvl: {"correct": 0, "total": 0} for lvl in level_range} for m in models}

    stats_with_fig, stats_no_fig, stats_all = init_stats(), init_stats(), init_stats()

    for model in models:
        for r in responses_by_model.get(model, []):
            final = parse_boxed_letter(_extract_content(r))
            p = prompts_by_id.get(r.get("id"))
            if p is None:
                continue
            try:
                lvl = int(p.get("level", 0))
            except Exception:
                continue
            if lvl not in level_range:
                continue
            contains_figure = bool(p.get("contains_latex_figure_in_question"))
            correct_opt = p.get("correct_answer")
            is_correct = (
                final is not None
                and correct_opt is not None
                and str(final).strip().upper() == str(correct_opt).strip().upper()
            )

            stats_all[model][lvl]["total"] += 1
            if is_correct:
                stats_all[model][lvl]["correct"] += 1
            bucket = stats_with_fig if contains_figure else stats_no_fig
            bucket[model][lvl]["total"] += 1
            if is_correct:
                bucket[model][lvl]["correct"] += 1

    def compute_acc(stats):
        out = {}
        for model in models:
            out[model] = {}
            for lvl in level_range:
                c, t = stats[model][lvl]["correct"], stats[model][lvl]["total"]
                out[model][lvl] = {"correct": c, "total": t, "accuracy": (c / t * 100.0) if t else None}
        return out

    return compute_acc(stats_all), compute_acc(stats_with_fig), compute_acc(stats_no_fig)


def compute_oe_accuracy(models, judge_responses, judge_prompts_by_model, level_range=range(1, 6)):
    """Compute open-ended accuracy stats from judge scores (0/1) per model and level."""
    def init_stats():
        return {m: {lvl: {"correct": 0, "total": 0} for lvl in level_range} for m in models}

    stats_with_fig, stats_no_fig, stats_all = init_stats(), init_stats(), init_stats()

    for model in models:
        lookup = judge_prompts_by_model.get(model, {})
        for r in judge_responses.get(model, []):
            p = lookup.get(r.get("id"))
            score = r.get("score")
            if p is None or score is None:
                continue
            try:
                lvl = int(p.get("level", 0))
            except Exception:
                continue
            if lvl not in level_range:
                continue
            contains_figure = bool(p.get("contains_latex_figure_in_question"))

            stats_all[model][lvl]["total"] += 1
            stats_all[model][lvl]["correct"] += score
            bucket = stats_with_fig if contains_figure else stats_no_fig
            bucket[model][lvl]["total"] += 1
            bucket[model][lvl]["correct"] += score

    def compute_acc(stats):
        out = {}
        for model in models:
            out[model] = {}
            for lvl in level_range:
                c, t = stats[model][lvl]["correct"], stats[model][lvl]["total"]
                out[model][lvl] = {"correct": c, "total": t, "accuracy": (c / t * 100.0) if t else None}
        return out

    return compute_acc(stats_all), compute_acc(stats_with_fig), compute_acc(stats_no_fig)


def format_acc_report(title, acc, models, level_range=range(1, 6)):
    lines = [f"=== {title} ==="]
    for model in sorted(models):
        lines.append("")
        lines.append(f"Model: {model}")
        for lvl in level_range:
            a = acc[model][lvl]
            t = a["total"]
            if not t:
                lines.append(f"  Level {lvl}: no data")
            else:
                lines.append(f"  Level {lvl}: {a['correct']}/{t} correct ({a['accuracy']:.2f}%)")
    lines.append("")
    return "\n".join(lines)


print("Funções de avaliação prontas: parse_boxed_letter, compute_mc_accuracy, compute_oe_accuracy, format_acc_report.")


## Parte 3 — Reproduzir os resultados já publicados

Esta parte recalcula os relatórios de acurácia em `results/accuracy-reports/` a partir dos prompts (`prompts/`, `judge-prompts/`) e respostas (`results/`, `judge-results/`) **já incluídos no repositório**. Não é necessária nenhuma chave de API para rodar esta parte.


### 3.1 — Múltipla escolha (execução única)

Equivalente a `evaluate-multiple-choice-answers.ipynb`.


In [ ]:
question_type = "multiple-choice"
dataset = "ptbr"  # "ptbr" ou "ptpt"
prompt_language = dataset  # "ptbr", "ptpt" ou "en" (ex.: "en" para os prompts em inglês do gpt-5)
run_dir = "first_run"  # "first_run" ou "second_run", ver results/multiple-choice/

models = {
    "claude-haiku-4.5", "deepseek-chat-v3.1", "gemini-2.5-flash", "gemma-3-27b-it",
    "gpt-5", "llama-3.3-70b-instruct", "qwen3-8b", "qwen3-14b", "qwen3-32b",
    "qwen3-30b-a3b", "qwen3-235b-a22b",
}


In [ ]:
with open(f"prompts/{dataset}-{question_type}-prompts-prompt-language-{prompt_language}.json", "r", encoding="utf-8") as f:
    prompts = json.load(f)

responses = {model: [] for model in models}
for model in models:
    path = f"results/{question_type}/{run_dir}/{model}-{dataset}-responses-prompt-language-{prompt_language}.json"
    if not os.path.exists(path):
        print(f"[aviso] arquivo não encontrado, pulando: {path}")
        continue
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                responses[model].append(json.loads(line))

print("Modelos com respostas carregadas:", sorted(m for m in models if responses[m]))


In [ ]:
prompts_by_id = {item["id"]: item for item in prompts}
acc_all, acc_with_fig, acc_no_fig = compute_mc_accuracy(models, responses, prompts_by_id)

report_text = "\n".join([
    format_acc_report("Overall (fig and no fig)", acc_all, models),
    format_acc_report("With figures only", acc_with_fig, models),
    format_acc_report("Without figures", acc_no_fig, models),
])

os.makedirs("results/accuracy-reports", exist_ok=True)
out_path = f"results/accuracy-reports/{dataset}-{question_type}-all-models-prompt-language-{prompt_language}-acc-report.txt"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(report_text)
print(f"Relatório salvo em: {out_path}")


### 3.2 — Múltipla escolha: variabilidade entre execuções

Equivalente a `evaluate-multiple-choice-answers-variability.ipynb`, com uma saída diferente do notebook original: em vez de um único relatório em texto, esta célula agrega a acurácia de várias execuções (`results/multiple-choice/first_run`, `second_run`, ...) e produz:

1. **Um arquivo JSON por execução** — `results/accuracy-reports/{dataset}-multiple-choice-first_run-acc.json`, `...-second_run-acc.json` etc. — com a acurácia bruta (`correct`/`total`/`accuracy`) por modelo e nível daquela rodada específica, separada em geral/com figura/sem figura.
2. **Um gráfico** — `results/accuracy-reports/{dataset}-multiple-choice-variability.png` — com a acurácia média entre execuções por nível (uma linha por modelo) e barra de erro = desvio padrão entre as rodadas, além de uma tabela-resumo na tela para conferência rápida.

In [ ]:
import glob
import statistics

question_type = "multiple-choice"
dataset = "ptbr"  # "ptbr" ou "ptpt"
prompt_language = dataset
models = {
    "llama-3.3-70b-instruct", "qwen3-8b", "qwen3-14b", "qwen3-32b",
    "qwen3-30b-a3b", "qwen3-235b-a22b", "gemma-3-27b-it",
}

with open(f"prompts/{dataset}-{question_type}-prompts-prompt-language-{prompt_language}.json", "r", encoding="utf-8") as f:
    prompts = json.load(f)
prompts_by_id = {item["id"]: item for item in prompts}

base_runs_dir = f"results/{question_type}"
run_dirs = sorted(d for d in glob.glob(f"{base_runs_dir}/*") if os.path.isdir(d))
print("Execuções detectadas:", run_dirs)

responses_by_run = {}
for run_dir_path in run_dirs:
    run_name = os.path.basename(run_dir_path)
    responses_by_run[run_name] = {model: [] for model in models}
    for model in models:
        file_path = f"{run_dir_path}/{model}-{dataset}-responses-prompt-language-{prompt_language}.json"
        if not os.path.exists(file_path):
            continue
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    responses_by_run[run_name][model].append(json.loads(line))

print("Respostas carregadas para execuções:", list(responses_by_run.keys()))


In [ ]:
LEVEL_RANGE = range(1, 5)

all_runs_acc_all, all_runs_acc_with, all_runs_acc_without = {}, {}, {}
for run_name, responses_for_run in responses_by_run.items():
    acc_all, acc_with, acc_without = compute_mc_accuracy(models, responses_for_run, prompts_by_id, level_range=LEVEL_RANGE)
    all_runs_acc_all[run_name] = acc_all
    all_runs_acc_with[run_name] = acc_with
    all_runs_acc_without[run_name] = acc_without

os.makedirs("results/accuracy-reports", exist_ok=True)


# --- 1) Um arquivo JSON por execução, com a acurácia bruta (não agregada) daquela rodada ---
def serialize_acc(acc):
    return {model: {str(lvl): v for lvl, v in lvls.items()} for model, lvls in acc.items()}


for run_name in responses_by_run:
    run_out = {
        "all": serialize_acc(all_runs_acc_all[run_name]),
        "with_fig": serialize_acc(all_runs_acc_with[run_name]),
        "no_fig": serialize_acc(all_runs_acc_without[run_name]),
    }
    run_path = f"results/accuracy-reports/{dataset}-{question_type}-{run_name}-acc.json"
    with open(run_path, "w", encoding="utf-8") as f:
        json.dump(run_out, f, ensure_ascii=False, indent=2)
    print("Salvo:", run_path)


# --- 2) Agregação (média ± desvio padrão entre execuções), usada no gráfico e no resumo ---
def aggregate_runs(run_dict, level_range=LEVEL_RANGE):
    out = {model: {lvl: {} for lvl in level_range} for model in models}
    for model in models:
        for lvl in level_range:
            values = [
                run_dict[r][model][lvl]["accuracy"]
                for r in run_dict
                if run_dict[r][model][lvl]["accuracy"] is not None
            ]
            if values:
                out[model][lvl] = {"mean": statistics.mean(values), "std": statistics.pstdev(values), "runs": values}
            else:
                out[model][lvl] = {"mean": None, "std": None, "runs": []}
    return out


agg_all = aggregate_runs(all_runs_acc_all)

# --- 3) Gráfico: acurácia média (± desvio padrão) por nível, uma linha por modelo ---
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))
for model in sorted(models):
    xs, ys, errs = [], [], []
    for lvl in LEVEL_RANGE:
        stats = agg_all[model][lvl]
        if stats["mean"] is None:
            continue
        xs.append(lvl)
        ys.append(stats["mean"])
        errs.append(stats["std"])
    if xs:
        ax.errorbar(xs, ys, yerr=errs, marker="o", capsize=4, label=model)

ax.set_xlabel("Nível")
ax.set_ylabel("Acurácia (%)")
ax.set_xticks(list(LEVEL_RANGE))
ax.set_ylim(0, 100)
ax.set_title(f"Variabilidade entre execuções ({', '.join(sorted(responses_by_run))}) — {dataset} multiple-choice")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, title="Modelo")
ax.grid(alpha=0.3)
fig.tight_layout()

plot_path = f"results/accuracy-reports/{dataset}-{question_type}-variability.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico salvo em:", plot_path)

# --- 4) Tabela-resumo na tela (média ± desvio padrão), só para conferência rápida ---
import pandas as pd
from IPython.display import display

summary_rows = []
for model in sorted(models):
    row = {"modelo": model}
    for lvl in LEVEL_RANGE:
        stats = agg_all[model][lvl]
        row[f"nível {lvl}"] = f"{stats['mean']:.2f} ± {stats['std']:.2f}" if stats["mean"] is not None else "—"
    summary_rows.append(row)
display(pd.DataFrame(summary_rows))

### 3.3 — Questões abertas (avaliação via juiz)

Equivalente a `evaluate-open-ended-answers.ipynb`. Usa as notas (0/1) atribuídas por um modelo-juiz em `judge-results/` para calcular acurácia.

> Correção em relação ao notebook original: cada modelo agora usa o seu próprio arquivo de `judge-prompts` para obter `level`/`contains_latex_figure_in_question` (o notebook original reaproveitava por engano apenas o último arquivo carregado no loop para todos os modelos).


In [ ]:
question_type = "open-ended"
dataset = "ptbr"  # "ptbr" ou "ptpt"
prompt_language = dataset
models = {
    "qwen3-8b", "qwen3-14b", "qwen3-32b", "qwen3-30b-a3b", "qwen3-235b-a22b",
    "deepseek-chat-v3.1", "gemma-3-27b-it", "llama-3.3-70b-instruct", "gpt-5",
    "gemini-2.5-flash", "claude-haiku-4.5",
}
judge = "kimi-k2-thinking"  # ver judge-results/ para outras opções (ex.: "qwen3-14b" para ptpt)

judge_responses = {model: [] for model in models}
for model in models:
    path = f"judge-results/{model}-{dataset}-judge-responses-prompt-language-{prompt_language}-judge-{judge}.json"
    if not os.path.exists(path):
        print(f"[aviso] não encontrado: {path}")
        continue
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                judge_responses[model].append(json.loads(line))

judge_prompts_by_model = {}
for model in models:
    path = f"judge-prompts/{model}-{dataset}-judge-prompts-prompt-language-{prompt_language}.json"
    if not os.path.exists(path):
        print(f"[aviso] não encontrado: {path}")
        continue
    with open(path, "r", encoding="utf-8") as f:
        judge_prompts_by_model[model] = {item["id"]: item for item in json.load(f)}

print("Modelos com respostas de juiz carregadas:", sorted(m for m in models if judge_responses[m]))


In [ ]:
for model in models:
    for response in judge_responses[model]:
        text = _extract_content(response)
        if text is None:
            print(f"[aviso] resposta do juiz ausente para id={response.get('id')} (modelo {model})")
            response["score"] = None
            response["explanation"] = None
            continue
        m_score = score_re.search(text)
        if not m_score:
            print(f"[aviso] score não encontrado para id={response.get('id')} (modelo {model})")
            response["score"] = None
            response["explanation"] = None
            continue
        m_expl = explanation_re.search(text)
        response["score"] = int(m_score.group(1))
        response["explanation"] = m_expl.group(1) if m_expl else None


In [ ]:
acc_all, acc_with_fig, acc_no_fig = compute_oe_accuracy(models, judge_responses, judge_prompts_by_model)

report_text = "\n".join([
    format_acc_report("Overall (fig and no fig)", acc_all, models),
    format_acc_report("With figures only", acc_with_fig, models),
    format_acc_report("Without figures", acc_no_fig, models),
])

out_path = f"results/accuracy-reports/{dataset}-{question_type}-all-models-prompt-language-{prompt_language}-judge-{judge}-acc-report.txt"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(report_text)
print("Relatório salvo em:", out_path)


## Parte 4 — Comparação com o artigo (MATH-PT)

Esta parte confronta, tabela por tabela, os números publicados no artigo que este repositório reproduz — **MATH-PT: A Math Reasoning Benchmark for European and Brazilian Portuguese** (Tiago Teixeira, Ana Carolina Erthal, Juan Belieni, Beatriz Canaverde, Diego Mesquita, Miguel Faria, Eliezer de Souza da Silva e André F. T. Martins — [deep-spin/math-benchmark](https://github.com/deep-spin/math-benchmark); dataset em [HuggingFace](https://huggingface.co/datasets/tiagoteixeira03/MATH-PT)) — com o que este notebook recalcula a partir dos arquivos já incluídos no repositório (`data/`, `prompts/`, `results/`, `judge-prompts/`, `judge-results/`).

O artigo avalia **13 modelos** (de fronteira e *open-weight*) em dois idiomas (**pt-PT**, Português Europeu, e **pt-BR**, Português do Brasil), dois tipos de questão (**MC** = múltipla escolha, **OE** = abertas) e **níveis de dificuldade** de 1 a 5 — correspondendo a categorias/séries escolares diferentes em Portugal (OPM, níveis 1–4, com questões com figura) e no Brasil (OBMEP/OBM/OMIF/ELLM/ITA, níveis 2–5, sem figuras). As Tabelas 1–4 do artigo cobrem, respectivamente: estatísticas do dataset, desempenho geral por nível, impacto de figuras (só pt-PT) e o efeito da escala dentro da família Qwen3.

Cada subseção abaixo:
1. Reproduz a tabela do artigo em Markdown (transcrita do PDF, "–" = combinação idioma/nível que não existe no artigo).
2. Recalcula a mesma métrica **neste notebook**, a partir dos dados já publicados no repositório — de forma independente do que ficou selecionado em `dataset`/`question_type` nas células da Parte 3 (a célula abaixo carrega os dois idiomas e os dois tipos de questão de uma vez).
3. Mostra lado a lado artigo × nosso × diferença (Δ, em pontos percentuais).

Assim como a Parte 3, esta parte não requer nenhuma chave de API: usa só os arquivos de prompts/respostas/julgamentos já commitados no repositório.

In [ ]:
# Recalcula, de forma independente da Parte 3, a acurácia para os dois idiomas
# (pt-PT, pt-BR) e os dois tipos de questão (múltipla escolha, questões abertas),
# reaproveitando as funções definidas na Parte 2.
import pandas as pd
from IPython.display import display
from collections import Counter

cmp_models = {
    "claude-haiku-4.5", "deepseek-chat-v3.1", "gemini-2.5-flash", "gemma-3-27b-it",
    "gpt-5", "llama-3.3-70b-instruct", "qwen3-8b", "qwen3-14b", "qwen3-32b",
    "qwen3-30b-a3b", "qwen3-235b-a22b",
}
cmp_judge = "kimi-k2-thinking"
cmp_run_dir = "first_run"  # mesma rodada usada na Parte 3.1

# cmp_acc[idioma][tipo] = (acc_all, acc_com_figura, acc_sem_figura), como retornado
# por compute_mc_accuracy / compute_oe_accuracy.
cmp_acc = {}

for cmp_dataset in ["ptpt", "ptbr"]:
    cmp_prompt_language = cmp_dataset

    # --- múltipla escolha ---
    with open(f"prompts/{cmp_dataset}-multiple-choice-prompts-prompt-language-{cmp_prompt_language}.json", "r", encoding="utf-8") as f:
        cmp_mc_prompts = json.load(f)
    cmp_mc_prompts_by_id = {item["id"]: item for item in cmp_mc_prompts}

    cmp_mc_responses = {model: [] for model in cmp_models}
    for model in cmp_models:
        path = f"results/multiple-choice/{cmp_run_dir}/{model}-{cmp_dataset}-responses-prompt-language-{cmp_prompt_language}.json"
        if not os.path.exists(path):
            print(f"[aviso] arquivo não encontrado, pulando: {path}")
            continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    cmp_mc_responses[model].append(json.loads(line))

    cmp_acc.setdefault(cmp_dataset, {})["mc"] = compute_mc_accuracy(cmp_models, cmp_mc_responses, cmp_mc_prompts_by_id)

    # --- questões abertas ---
    cmp_judge_responses = {model: [] for model in cmp_models}
    for model in cmp_models:
        path = f"judge-results/{model}-{cmp_dataset}-judge-responses-prompt-language-{cmp_prompt_language}-judge-{cmp_judge}.json"
        if not os.path.exists(path):
            print(f"[aviso] não encontrado: {path}")
            continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    cmp_judge_responses[model].append(json.loads(line))

    cmp_judge_prompts_by_model = {}
    for model in cmp_models:
        path = f"judge-prompts/{model}-{cmp_dataset}-judge-prompts-prompt-language-{cmp_prompt_language}.json"
        if not os.path.exists(path):
            continue
        with open(path, "r", encoding="utf-8") as f:
            cmp_judge_prompts_by_model[model] = {item["id"]: item for item in json.load(f)}

    for model in cmp_models:
        for response in cmp_judge_responses[model]:
            text = _extract_content(response)
            if text is None:
                response["score"] = None
                continue
            m_score = score_re.search(text)
            response["score"] = int(m_score.group(1)) if m_score else None

    cmp_acc[cmp_dataset]["oe"] = compute_oe_accuracy(cmp_models, cmp_judge_responses, cmp_judge_prompts_by_model)


def cmp_get(dataset, qtype, fig, model, level):
    """fig: "all" | "fig" | "nofig". Retorna a acurácia (%) arredondada, ou None se não houver dados."""
    idx = {"all": 0, "fig": 1, "nofig": 2}[fig]
    acc = cmp_acc[dataset][qtype][idx]
    entry = acc.get(model, {}).get(level)
    if not entry or entry["accuracy"] is None:
        return None
    return round(entry["accuracy"], 2)


print("Acurácias recalculadas para:", {k: list(v.keys()) for k, v in cmp_acc.items()})

### 4.1 — Tabela 1 do artigo: estatísticas do dataset

Tabela 1 do artigo (número de questões por país, competição, nível e tipo):

| País | Competição | Nível | MC | OE | Total |
|---|---|---|---|---|---|
| Portugal | OPM | 1 | 78 | 34 | 112 |
| Portugal | OPM | 2 | 169 | 96 | 265 |
| Portugal | OPM | 3 | 143 | 193 | 336 |
| Portugal | OPM | 4 | 29 | 221 | 250 |
| Brasil | OBMEP | 2 | 118 | – | 118 |
| Brasil | OBMEP | 3 | 140 | – | 140 |
| Brasil | OBMEP | 4 | 167 | – | 167 |
| Brasil | OBM | 2 | – | 32 | 32 |
| Brasil | OBM | 3 | – | 58 | 58 |
| Brasil | OBM | 4 | – | 38 | 38 |
| Brasil | OMIF | 4 | 103 | – | 103 |
| Brasil | ELLM | 5 | 53 | – | 53 |
| Brasil | ITA | 5 | 57 | – | 57 |
| **Total** | | | **1.057** | **672** | **1.729** |

Os arquivos `data/*.json` não preservam a granularidade por competição (OPM/OBMEP/OBM/OMIF/ELLM/ITA), só o par idioma+nível — a comparação abaixo agrega as linhas do artigo por (idioma, nível), que é a granularidade máxima reproduzível a partir dos dados publicados.

In [ ]:
# Agregações do artigo por (idioma, nível), obtidas somando as linhas da Tabela 1 por país
paper_table1 = {
    ("ptpt", 1): {"mc": 78, "oe": 34},
    ("ptpt", 2): {"mc": 169, "oe": 96},
    ("ptpt", 3): {"mc": 143, "oe": 193},
    ("ptpt", 4): {"mc": 29, "oe": 221},
    ("ptbr", 2): {"mc": 118, "oe": 32},
    ("ptbr", 3): {"mc": 140, "oe": 58},
    ("ptbr", 4): {"mc": 167 + 103, "oe": 38},  # OBMEP (MC) + OMIF (MC) + OBM (OE)
    ("ptbr", 5): {"mc": 53 + 57, "oe": 0},     # ELLM (MC) + ITA (MC)
}

rows = []
for dataset in ["ptpt", "ptbr"]:
    for qtype, key in [("multiple-choice", "mc"), ("open-ended", "oe")]:
        _file_qtype = "multiple_choice" if qtype == "multiple-choice" else "open_ended"
        with open(f"data/{dataset}_{_file_qtype}_test.json", "r", encoding="utf-8") as f:
            qa = json.load(f)
        counts = Counter(int(item["level"]) for item in qa if item.get("level") is not None)
        for level in range(1, 6):
            paper_n = paper_table1.get((dataset, level), {}).get(key, 0)
            our_n = counts.get(level, 0)
            if paper_n == 0 and our_n == 0:
                continue
            rows.append({
                "idioma": dataset, "nível": level, "tipo": qtype,
                "artigo": paper_n, "notebook": our_n, "Δ": our_n - paper_n,
            })

df_table1 = pd.DataFrame(rows)
display(df_table1)
print("Total de questões — artigo: 1.729 | notebook:", df_table1["notebook"].sum())

### 4.2 — Tabela 2 do artigo: desempenho geral por nível

Tabela 2 do artigo — acurácia (%) de múltipla escolha (MC) e questões abertas (OE), por nível de dificuldade, em pt-PT e pt-BR:

| Modelo | Nível | pt-PT MC | pt-PT OE | pt-BR MC | pt-BR OE |
|---|---|---|---|---|---|
| GPT-5 | L1 | 96.15 | 100.00 | – | – |
| GPT-5 | L2 | 94.67 | 89.58 | 90.68 | 90.62 |
| GPT-5 | L3 | 96.50 | 92.75 | 92.86 | 91.38 |
| GPT-5 | L4 | 96.55 | 85.52 | 93.33 | 89.47 |
| GPT-5 | L5 | – | – | 90.00 | – |
| Qwen3-235B-A22B | L1 | 91.03 | 88.24 | – | – |
| Qwen3-235B-A22B | L2 | 86.98 | 79.17 | 88.98 | 84.38 |
| Qwen3-235B-A22B | L3 | 88.11 | 81.35 | 92.86 | 93.10 |
| Qwen3-235B-A22B | L4 | 93.10 | 78.73 | 91.48 | 86.84 |
| Qwen3-235B-A22B | L5 | – | – | 89.09 | – |
| Llama-3.3-70b-Instruct | L1 | 41.03 | 38.24 | – | – |
| Llama-3.3-70b-Instruct | L2 | 32.54 | 17.71 | 35.59 | 34.38 |
| Llama-3.3-70b-Instruct | L3 | 23.78 | 18.13 | 35.00 | 34.48 |
| Llama-3.3-70b-Instruct | L4 | 24.14 | 20.36 | 32.96 | 18.42 |
| Llama-3.3-70b-Instruct | L5 | – | – | 31.82 | – |
| Gemma-3-27b-it | L1 | 57.69 | 58.82 | – | – |
| Gemma-3-27b-it | L2 | 49.11 | 30.21 | 66.95 | 65.62 |
| Gemma-3-27b-it | L3 | 48.95 | 33.16 | 69.29 | 67.24 |
| Gemma-3-27b-it | L4 | 58.62 | 27.60 | 64.81 | 52.63 |
| Gemma-3-27b-it | L5 | – | – | 61.82 | – |
| Deepseek-chat-v3.1 | L1 | 87.18 | 73.53 | – | – |
| Deepseek-chat-v3.1 | L2 | 84.02 | 67.71 | 83.05 | 78.12 |
| Deepseek-chat-v3.1 | L3 | 88.81 | 72.02 | 91.43 | 86.21 |
| Deepseek-chat-v3.1 | L4 | 79.31 | 66.52 | 89.26 | 81.58 |
| Deepseek-chat-v3.1 | L5 | – | – | 86.36 | – |
| Claude-Haiku-4.5 | L1 | 80.77 | 67.65 | – | – |
| Claude-Haiku-4.5 | L2 | 74.56 | 47.92 | 83.90 | 68.75 |
| Claude-Haiku-4.5 | L3 | 76.92 | 56.48 | 84.29 | 70.69 |
| Claude-Haiku-4.5 | L4 | 68.97 | 47.51 | 81.48 | 60.53 |
| Claude-Haiku-4.5 | L5 | – | – | 81.82 | – |
| Gemini-2.5-Flash | L1 | 83.33 | 88.24 | – | – |
| Gemini-2.5-Flash | L2 | 75.74 | 72.92 | 83.90 | 84.38 |
| Gemini-2.5-Flash | L3 | 83.92 | 74.61 | 88.57 | 91.38 |
| Gemini-2.5-Flash | L4 | 79.31 | 71.04 | 85.93 | 71.05 |
| Gemini-2.5-Flash | L5 | – | – | 89.09 | – |

A célula abaixo recalcula os mesmos números a partir de `results/multiple-choice/first_run` e `judge-results/` e mostra artigo × nosso × Δ (em pontos percentuais) lado a lado.

In [ ]:
# Números da Tabela 2 do artigo, por (modelo, nível). None = célula "N/A" no artigo
# (combinação idioma/nível que não existe naquele país).
paper_table2 = {
    ("gpt-5", 1): {"ptpt_mc": 96.15, "ptpt_oe": 100.00, "ptbr_mc": None, "ptbr_oe": None},
    ("gpt-5", 2): {"ptpt_mc": 94.67, "ptpt_oe": 89.58, "ptbr_mc": 90.68, "ptbr_oe": 90.62},
    ("gpt-5", 3): {"ptpt_mc": 96.50, "ptpt_oe": 92.75, "ptbr_mc": 92.86, "ptbr_oe": 91.38},
    ("gpt-5", 4): {"ptpt_mc": 96.55, "ptpt_oe": 85.52, "ptbr_mc": 93.33, "ptbr_oe": 89.47},
    ("gpt-5", 5): {"ptpt_mc": None, "ptpt_oe": None, "ptbr_mc": 90.00, "ptbr_oe": None},
    ("qwen3-235b-a22b", 1): {"ptpt_mc": 91.03, "ptpt_oe": 88.24, "ptbr_mc": None, "ptbr_oe": None},
    ("qwen3-235b-a22b", 2): {"ptpt_mc": 86.98, "ptpt_oe": 79.17, "ptbr_mc": 88.98, "ptbr_oe": 84.38},
    ("qwen3-235b-a22b", 3): {"ptpt_mc": 88.11, "ptpt_oe": 81.35, "ptbr_mc": 92.86, "ptbr_oe": 93.10},
    ("qwen3-235b-a22b", 4): {"ptpt_mc": 93.10, "ptpt_oe": 78.73, "ptbr_mc": 91.48, "ptbr_oe": 86.84},
    ("qwen3-235b-a22b", 5): {"ptpt_mc": None, "ptpt_oe": None, "ptbr_mc": 89.09, "ptbr_oe": None},
    ("llama-3.3-70b-instruct", 1): {"ptpt_mc": 41.03, "ptpt_oe": 38.24, "ptbr_mc": None, "ptbr_oe": None},
    ("llama-3.3-70b-instruct", 2): {"ptpt_mc": 32.54, "ptpt_oe": 17.71, "ptbr_mc": 35.59, "ptbr_oe": 34.38},
    ("llama-3.3-70b-instruct", 3): {"ptpt_mc": 23.78, "ptpt_oe": 18.13, "ptbr_mc": 35.00, "ptbr_oe": 34.48},
    ("llama-3.3-70b-instruct", 4): {"ptpt_mc": 24.14, "ptpt_oe": 20.36, "ptbr_mc": 32.96, "ptbr_oe": 18.42},
    ("llama-3.3-70b-instruct", 5): {"ptpt_mc": None, "ptpt_oe": None, "ptbr_mc": 31.82, "ptbr_oe": None},
    ("gemma-3-27b-it", 1): {"ptpt_mc": 57.69, "ptpt_oe": 58.82, "ptbr_mc": None, "ptbr_oe": None},
    ("gemma-3-27b-it", 2): {"ptpt_mc": 49.11, "ptpt_oe": 30.21, "ptbr_mc": 66.95, "ptbr_oe": 65.62},
    ("gemma-3-27b-it", 3): {"ptpt_mc": 48.95, "ptpt_oe": 33.16, "ptbr_mc": 69.29, "ptbr_oe": 67.24},
    ("gemma-3-27b-it", 4): {"ptpt_mc": 58.62, "ptpt_oe": 27.60, "ptbr_mc": 64.81, "ptbr_oe": 52.63},
    ("gemma-3-27b-it", 5): {"ptpt_mc": None, "ptpt_oe": None, "ptbr_mc": 61.82, "ptbr_oe": None},
    ("deepseek-chat-v3.1", 1): {"ptpt_mc": 87.18, "ptpt_oe": 73.53, "ptbr_mc": None, "ptbr_oe": None},
    ("deepseek-chat-v3.1", 2): {"ptpt_mc": 84.02, "ptpt_oe": 67.71, "ptbr_mc": 83.05, "ptbr_oe": 78.12},
    ("deepseek-chat-v3.1", 3): {"ptpt_mc": 88.81, "ptpt_oe": 72.02, "ptbr_mc": 91.43, "ptbr_oe": 86.21},
    ("deepseek-chat-v3.1", 4): {"ptpt_mc": 79.31, "ptpt_oe": 66.52, "ptbr_mc": 89.26, "ptbr_oe": 81.58},
    ("deepseek-chat-v3.1", 5): {"ptpt_mc": None, "ptpt_oe": None, "ptbr_mc": 86.36, "ptbr_oe": None},
    ("claude-haiku-4.5", 1): {"ptpt_mc": 80.77, "ptpt_oe": 67.65, "ptbr_mc": None, "ptbr_oe": None},
    ("claude-haiku-4.5", 2): {"ptpt_mc": 74.56, "ptpt_oe": 47.92, "ptbr_mc": 83.90, "ptbr_oe": 68.75},
    ("claude-haiku-4.5", 3): {"ptpt_mc": 76.92, "ptpt_oe": 56.48, "ptbr_mc": 84.29, "ptbr_oe": 70.69},
    ("claude-haiku-4.5", 4): {"ptpt_mc": 68.97, "ptpt_oe": 47.51, "ptbr_mc": 81.48, "ptbr_oe": 60.53},
    ("claude-haiku-4.5", 5): {"ptpt_mc": None, "ptpt_oe": None, "ptbr_mc": 81.82, "ptbr_oe": None},
    ("gemini-2.5-flash", 1): {"ptpt_mc": 83.33, "ptpt_oe": 88.24, "ptbr_mc": None, "ptbr_oe": None},
    ("gemini-2.5-flash", 2): {"ptpt_mc": 75.74, "ptpt_oe": 72.92, "ptbr_mc": 83.90, "ptbr_oe": 84.38},
    ("gemini-2.5-flash", 3): {"ptpt_mc": 83.92, "ptpt_oe": 74.61, "ptbr_mc": 88.57, "ptbr_oe": 91.38},
    ("gemini-2.5-flash", 4): {"ptpt_mc": 79.31, "ptpt_oe": 71.04, "ptbr_mc": 85.93, "ptbr_oe": 71.05},
    ("gemini-2.5-flash", 5): {"ptpt_mc": None, "ptpt_oe": None, "ptbr_mc": 89.09, "ptbr_oe": None},
}

rows = []
for (model, level), paper_vals in sorted(paper_table2.items(), key=lambda kv: (kv[0][0], kv[0][1])):
    our_vals = {
        "ptpt_mc": cmp_get("ptpt", "mc", "all", model, level),
        "ptpt_oe": cmp_get("ptpt", "oe", "all", model, level),
        "ptbr_mc": cmp_get("ptbr", "mc", "all", model, level),
        "ptbr_oe": cmp_get("ptbr", "oe", "all", model, level),
    }
    row = {"modelo": model, "nível": level}
    for key in ["ptpt_mc", "ptpt_oe", "ptbr_mc", "ptbr_oe"]:
        p, o = paper_vals[key], our_vals[key]
        row[f"{key} (artigo)"] = p
        row[f"{key} (nosso)"] = o
        row[f"{key} (Δ)"] = None if (p is None or o is None) else round(o - p, 2)
    rows.append(row)

df_table2 = pd.DataFrame(rows)
display(df_table2)

deltas = [abs(v) for r in rows for k, v in r.items() if k.endswith("(Δ)") and v is not None]
print(f"Tabela 2 — n={len(deltas)} comparações | maior |Δ| = {max(deltas):.2f} pp | Δ médio = {sum(deltas)/len(deltas):.4f} pp")

### 4.3 — Tabela 3 do artigo: impacto de figuras (pt-PT)

O pt-BR não tem questões com figura (ver Tabela 1), então esta comparação é só para pt-PT. Tabela 3 do artigo — acurácia (%) separada por presença de figura na questão, níveis 1–4:

| Modelo | Nível | MC sem figura | MC com figura | OE sem figura | OE com figura |
|---|---|---|---|---|---|
| GPT-5 | L1 | 100.00 | 88.00 | 100.00 | 100.00 |
| GPT-5 | L2 | 96.52 | 90.74 | 88.46 | 90.91 |
| GPT-5 | L3 | 97.92 | 93.62 | 94.35 | 89.86 |
| GPT-5 | L4 | 100.00 | 87.50 | 83.83 | 90.74 |
| Qwen3-235B-A22B | L1 | 98.11 | 76.00 | 100.00 | 73.33 |
| Qwen3-235B-A22B | L2 | 90.43 | 79.63 | 82.69 | 75.00 |
| Qwen3-235B-A22B | L3 | 95.83 | 72.34 | 86.29 | 72.46 |
| Qwen3-235B-A22B | L4 | 95.24 | 87.50 | 80.24 | 74.07 |
| Llama-3.3-70b-Instruct | L1 | 49.06 | 24.00 | 63.16 | 6.67 |
| Llama-3.3-70b-Instruct | L2 | 40.00 | 16.67 | 21.15 | 13.64 |
| Llama-3.3-70b-Instruct | L3 | 32.29 | 6.38 | 21.77 | 11.59 |
| Llama-3.3-70b-Instruct | L4 | 28.57 | 12.50 | 19.76 | 22.22 |
| Gemma-3-27b-it | L1 | 66.04 | 40.00 | 73.68 | 40.00 |
| Gemma-3-27b-it | L2 | 59.13 | 27.78 | 42.31 | 15.91 |
| Gemma-3-27b-it | L3 | 58.33 | 29.79 | 33.87 | 31.88 |
| Gemma-3-27b-it | L4 | 66.67 | 37.50 | 25.75 | 33.33 |
| Deepseek-chat-v3.1 | L1 | 98.11 | 64.00 | 100.00 | 40.00 |
| Deepseek-chat-v3.1 | L2 | 94.78 | 61.11 | 73.08 | 61.36 |
| Deepseek-chat-v3.1 | L3 | 94.79 | 76.60 | 76.61 | 63.77 |
| Deepseek-chat-v3.1 | L4 | 80.95 | 75.00 | 68.26 | 61.11 |
| Claude-Haiku-4.5 | L1 | 94.34 | 52.00 | 89.47 | 40.00 |
| Claude-Haiku-4.5 | L2 | 84.35 | 53.70 | 57.69 | 36.36 |
| Claude-Haiku-4.5 | L3 | 83.33 | 63.83 | 61.29 | 47.83 |
| Claude-Haiku-4.5 | L4 | 76.19 | 50.00 | 47.90 | 46.30 |
| Gemini-2.5-Flash | L1 | 94.34 | 60.00 | 94.74 | 80.00 |
| Gemini-2.5-Flash | L2 | 84.35 | 57.41 | 78.85 | 65.91 |
| Gemini-2.5-Flash | L3 | 86.46 | 78.72 | 83.06 | 59.42 |
| Gemini-2.5-Flash | L4 | 80.95 | 75.00 | 71.86 | 68.52 |

In [ ]:
paper_table3 = {
    ("gpt-5", 1): {"mc_nofig": 100.00, "mc_fig": 88.00, "oe_nofig": 100.00, "oe_fig": 100.00},
    ("gpt-5", 2): {"mc_nofig": 96.52, "mc_fig": 90.74, "oe_nofig": 88.46, "oe_fig": 90.91},
    ("gpt-5", 3): {"mc_nofig": 97.92, "mc_fig": 93.62, "oe_nofig": 94.35, "oe_fig": 89.86},
    ("gpt-5", 4): {"mc_nofig": 100.00, "mc_fig": 87.50, "oe_nofig": 83.83, "oe_fig": 90.74},
    ("qwen3-235b-a22b", 1): {"mc_nofig": 98.11, "mc_fig": 76.00, "oe_nofig": 100.00, "oe_fig": 73.33},
    ("qwen3-235b-a22b", 2): {"mc_nofig": 90.43, "mc_fig": 79.63, "oe_nofig": 82.69, "oe_fig": 75.00},
    ("qwen3-235b-a22b", 3): {"mc_nofig": 95.83, "mc_fig": 72.34, "oe_nofig": 86.29, "oe_fig": 72.46},
    ("qwen3-235b-a22b", 4): {"mc_nofig": 95.24, "mc_fig": 87.50, "oe_nofig": 80.24, "oe_fig": 74.07},
    ("llama-3.3-70b-instruct", 1): {"mc_nofig": 49.06, "mc_fig": 24.00, "oe_nofig": 63.16, "oe_fig": 6.67},
    ("llama-3.3-70b-instruct", 2): {"mc_nofig": 40.00, "mc_fig": 16.67, "oe_nofig": 21.15, "oe_fig": 13.64},
    ("llama-3.3-70b-instruct", 3): {"mc_nofig": 32.29, "mc_fig": 6.38, "oe_nofig": 21.77, "oe_fig": 11.59},
    ("llama-3.3-70b-instruct", 4): {"mc_nofig": 28.57, "mc_fig": 12.50, "oe_nofig": 19.76, "oe_fig": 22.22},
    ("gemma-3-27b-it", 1): {"mc_nofig": 66.04, "mc_fig": 40.00, "oe_nofig": 73.68, "oe_fig": 40.00},
    ("gemma-3-27b-it", 2): {"mc_nofig": 59.13, "mc_fig": 27.78, "oe_nofig": 42.31, "oe_fig": 15.91},
    ("gemma-3-27b-it", 3): {"mc_nofig": 58.33, "mc_fig": 29.79, "oe_nofig": 33.87, "oe_fig": 31.88},
    ("gemma-3-27b-it", 4): {"mc_nofig": 66.67, "mc_fig": 37.50, "oe_nofig": 25.75, "oe_fig": 33.33},
    ("deepseek-chat-v3.1", 1): {"mc_nofig": 98.11, "mc_fig": 64.00, "oe_nofig": 100.00, "oe_fig": 40.00},
    ("deepseek-chat-v3.1", 2): {"mc_nofig": 94.78, "mc_fig": 61.11, "oe_nofig": 73.08, "oe_fig": 61.36},
    ("deepseek-chat-v3.1", 3): {"mc_nofig": 94.79, "mc_fig": 76.60, "oe_nofig": 76.61, "oe_fig": 63.77},
    ("deepseek-chat-v3.1", 4): {"mc_nofig": 80.95, "mc_fig": 75.00, "oe_nofig": 68.26, "oe_fig": 61.11},
    ("claude-haiku-4.5", 1): {"mc_nofig": 94.34, "mc_fig": 52.00, "oe_nofig": 89.47, "oe_fig": 40.00},
    ("claude-haiku-4.5", 2): {"mc_nofig": 84.35, "mc_fig": 53.70, "oe_nofig": 57.69, "oe_fig": 36.36},
    ("claude-haiku-4.5", 3): {"mc_nofig": 83.33, "mc_fig": 63.83, "oe_nofig": 61.29, "oe_fig": 47.83},
    ("claude-haiku-4.5", 4): {"mc_nofig": 76.19, "mc_fig": 50.00, "oe_nofig": 47.90, "oe_fig": 46.30},
    ("gemini-2.5-flash", 1): {"mc_nofig": 94.34, "mc_fig": 60.00, "oe_nofig": 94.74, "oe_fig": 80.00},
    ("gemini-2.5-flash", 2): {"mc_nofig": 84.35, "mc_fig": 57.41, "oe_nofig": 78.85, "oe_fig": 65.91},
    ("gemini-2.5-flash", 3): {"mc_nofig": 86.46, "mc_fig": 78.72, "oe_nofig": 83.06, "oe_fig": 59.42},
    ("gemini-2.5-flash", 4): {"mc_nofig": 80.95, "mc_fig": 75.00, "oe_nofig": 71.86, "oe_fig": 68.52},
}

rows = []
for (model, level), paper_vals in sorted(paper_table3.items(), key=lambda kv: (kv[0][0], kv[0][1])):
    our_vals = {
        "mc_nofig": cmp_get("ptpt", "mc", "nofig", model, level),
        "mc_fig": cmp_get("ptpt", "mc", "fig", model, level),
        "oe_nofig": cmp_get("ptpt", "oe", "nofig", model, level),
        "oe_fig": cmp_get("ptpt", "oe", "fig", model, level),
    }
    row = {"modelo": model, "nível": level}
    for key in ["mc_nofig", "mc_fig", "oe_nofig", "oe_fig"]:
        p, o = paper_vals[key], our_vals[key]
        row[f"{key} (artigo)"] = p
        row[f"{key} (nosso)"] = o
        row[f"{key} (Δ)"] = None if (p is None or o is None) else round(o - p, 2)
    rows.append(row)

df_table3 = pd.DataFrame(rows)
display(df_table3)

deltas3 = [abs(v) for r in rows for k, v in r.items() if k.endswith("(Δ)") and v is not None]
print(f"Tabela 3 — n={len(deltas3)} comparações | maior |Δ| = {max(deltas3):.2f} pp | Δ médio = {sum(deltas3)/len(deltas3):.4f} pp")

### 4.4 — Tabela 4 do artigo: efeito da escala no Qwen3

Tabela 4 do artigo — acurácia (%) para os cinco tamanhos da família Qwen3 avaliados, em pt-PT e pt-BR, MC e OE:

| Modelo | Nível | pt-PT MC | pt-PT OE | pt-BR MC | pt-BR OE |
|---|---|---|---|---|---|
| Qwen3-8B | L1 | 84.62 | 82.35 | – | – |
| Qwen3-8B | L2 | 76.33 | 58.33 | 87.29 | 75.00 |
| Qwen3-8B | L3 | 78.32 | 61.14 | 87.14 | 72.41 |
| Qwen3-8B | L4 | 65.52 | 50.23 | 81.48 | 52.63 |
| Qwen3-8B | L5 | – | – | 76.36 | – |
| Qwen3-14B | L1 | 88.46 | 82.35 | – | – |
| Qwen3-14B | L2 | 89.35 | 73.96 | 90.68 | 81.25 |
| Qwen3-14B | L3 | 86.71 | 76.68 | 90.00 | 84.48 |
| Qwen3-14B | L4 | 82.76 | 71.04 | 88.15 | 78.95 |
| Qwen3-14B | L5 | – | – | 90.91 | – |
| Qwen3-32B | L1 | 88.46 | 79.41 | – | – |
| Qwen3-32B | L2 | 83.43 | 70.83 | 88.98 | 84.38 |
| Qwen3-32B | L3 | 85.31 | 73.06 | 89.29 | 82.76 |
| Qwen3-32B | L4 | 93.10 | 65.16 | 85.19 | 76.32 |
| Qwen3-32B | L5 | – | – | 87.27 | – |
| Qwen3-30B-A3B | L1 | 82.05 | 82.35 | – | – |
| Qwen3-30B-A3B | L2 | 82.84 | 64.58 | 82.20 | 75.00 |
| Qwen3-30B-A3B | L3 | 86.71 | 63.73 | 89.29 | 74.14 |
| Qwen3-30B-A3B | L4 | 82.76 | 55.20 | 84.44 | 50.00 |
| Qwen3-30B-A3B | L5 | – | – | 80.00 | – |
| Qwen3-235B-A22B | L1 | 91.03 | 88.24 | – | – |
| Qwen3-235B-A22B | L2 | 86.98 | 79.17 | 88.98 | 84.38 |
| Qwen3-235B-A22B | L3 | 88.11 | 81.35 | 92.86 | 93.10 |
| Qwen3-235B-A22B | L4 | 93.10 | 78.73 | 91.48 | 86.84 |
| Qwen3-235B-A22B | L5 | – | – | 89.09 | – |

In [ ]:
paper_table4 = {
    ("qwen3-8b", 1): {"ptpt_mc": 84.62, "ptpt_oe": 82.35, "ptbr_mc": None, "ptbr_oe": None},
    ("qwen3-8b", 2): {"ptpt_mc": 76.33, "ptpt_oe": 58.33, "ptbr_mc": 87.29, "ptbr_oe": 75.00},
    ("qwen3-8b", 3): {"ptpt_mc": 78.32, "ptpt_oe": 61.14, "ptbr_mc": 87.14, "ptbr_oe": 72.41},
    ("qwen3-8b", 4): {"ptpt_mc": 65.52, "ptpt_oe": 50.23, "ptbr_mc": 81.48, "ptbr_oe": 52.63},
    ("qwen3-8b", 5): {"ptpt_mc": None, "ptpt_oe": None, "ptbr_mc": 76.36, "ptbr_oe": None},
    ("qwen3-14b", 1): {"ptpt_mc": 88.46, "ptpt_oe": 82.35, "ptbr_mc": None, "ptbr_oe": None},
    ("qwen3-14b", 2): {"ptpt_mc": 89.35, "ptpt_oe": 73.96, "ptbr_mc": 90.68, "ptbr_oe": 81.25},
    ("qwen3-14b", 3): {"ptpt_mc": 86.71, "ptpt_oe": 76.68, "ptbr_mc": 90.00, "ptbr_oe": 84.48},
    ("qwen3-14b", 4): {"ptpt_mc": 82.76, "ptpt_oe": 71.04, "ptbr_mc": 88.15, "ptbr_oe": 78.95},
    ("qwen3-14b", 5): {"ptpt_mc": None, "ptpt_oe": None, "ptbr_mc": 90.91, "ptbr_oe": None},
    ("qwen3-32b", 1): {"ptpt_mc": 88.46, "ptpt_oe": 79.41, "ptbr_mc": None, "ptbr_oe": None},
    ("qwen3-32b", 2): {"ptpt_mc": 83.43, "ptpt_oe": 70.83, "ptbr_mc": 88.98, "ptbr_oe": 84.38},
    ("qwen3-32b", 3): {"ptpt_mc": 85.31, "ptpt_oe": 73.06, "ptbr_mc": 89.29, "ptbr_oe": 82.76},
    ("qwen3-32b", 4): {"ptpt_mc": 93.10, "ptpt_oe": 65.16, "ptbr_mc": 85.19, "ptbr_oe": 76.32},
    ("qwen3-32b", 5): {"ptpt_mc": None, "ptpt_oe": None, "ptbr_mc": 87.27, "ptbr_oe": None},
    ("qwen3-30b-a3b", 1): {"ptpt_mc": 82.05, "ptpt_oe": 82.35, "ptbr_mc": None, "ptbr_oe": None},
    ("qwen3-30b-a3b", 2): {"ptpt_mc": 82.84, "ptpt_oe": 64.58, "ptbr_mc": 82.20, "ptbr_oe": 75.00},
    ("qwen3-30b-a3b", 3): {"ptpt_mc": 86.71, "ptpt_oe": 63.73, "ptbr_mc": 89.29, "ptbr_oe": 74.14},
    ("qwen3-30b-a3b", 4): {"ptpt_mc": 82.76, "ptpt_oe": 55.20, "ptbr_mc": 84.44, "ptbr_oe": 50.00},
    ("qwen3-30b-a3b", 5): {"ptpt_mc": None, "ptpt_oe": None, "ptbr_mc": 80.00, "ptbr_oe": None},
    ("qwen3-235b-a22b", 1): {"ptpt_mc": 91.03, "ptpt_oe": 88.24, "ptbr_mc": None, "ptbr_oe": None},
    ("qwen3-235b-a22b", 2): {"ptpt_mc": 86.98, "ptpt_oe": 79.17, "ptbr_mc": 88.98, "ptbr_oe": 84.38},
    ("qwen3-235b-a22b", 3): {"ptpt_mc": 88.11, "ptpt_oe": 81.35, "ptbr_mc": 92.86, "ptbr_oe": 93.10},
    ("qwen3-235b-a22b", 4): {"ptpt_mc": 93.10, "ptpt_oe": 78.73, "ptbr_mc": 91.48, "ptbr_oe": 86.84},
    ("qwen3-235b-a22b", 5): {"ptpt_mc": None, "ptpt_oe": None, "ptbr_mc": 89.09, "ptbr_oe": None},
}

qwen_size_order = {"qwen3-8b": 8, "qwen3-14b": 14, "qwen3-32b": 32, "qwen3-30b-a3b": 30, "qwen3-235b-a22b": 235}

rows = []
for (model, level), paper_vals in sorted(paper_table4.items(), key=lambda kv: (qwen_size_order[kv[0][0]], kv[0][1])):
    our_vals = {
        "ptpt_mc": cmp_get("ptpt", "mc", "all", model, level),
        "ptpt_oe": cmp_get("ptpt", "oe", "all", model, level),
        "ptbr_mc": cmp_get("ptbr", "mc", "all", model, level),
        "ptbr_oe": cmp_get("ptbr", "oe", "all", model, level),
    }
    row = {"modelo": model, "nível": level}
    for key in ["ptpt_mc", "ptpt_oe", "ptbr_mc", "ptbr_oe"]:
        p, o = paper_vals[key], our_vals[key]
        row[f"{key} (artigo)"] = p
        row[f"{key} (nosso)"] = o
        row[f"{key} (Δ)"] = None if (p is None or o is None) else round(o - p, 2)
    rows.append(row)

df_table4 = pd.DataFrame(rows)
display(df_table4)

deltas4 = [abs(v) for r in rows for k, v in r.items() if k.endswith("(Δ)") and v is not None]
print(f"Tabela 4 — n={len(deltas4)} comparações | maior |Δ| = {max(deltas4):.2f} pp | Δ médio = {sum(deltas4)/len(deltas4):.4f} pp")

### 4.5 — Interpretação didática dos resultados

**A reprodução bate exatamente com o artigo.** Rodando as células acima sobre os dados já publicados neste repositório (`results/multiple-choice/first_run`, `judge-results/`), as comparações feitas nas Tabelas 2, 3 e 4 (105 + 112 + 75 = 292 células numéricas) dão **Δ = 0,00 ponto percentual em todas elas** — nenhuma discrepância, nem de arredondamento. A Tabela 1 (contagem de questões por idioma/nível/tipo) também bate perfeitamente com os totais do artigo (1.729 questões: 1.057 MC + 672 OE).

**Por que a reprodução é exata (e quando ela deixaria de ser):** a Parte 3 deste notebook não faz nenhuma chamada de API — ela relê os arquivos de respostas de modelo (`results/`) e de julgamento (`judge-results/`) que já foram gerados uma vez e commitados no repositório. A extração da letra final (`\boxed{...}`) para múltipla escolha é uma operação puramente determinística de regex sobre texto já salvo, e a nota das questões abertas também já veio pronta do juiz (Kimi K2 Thinking) em `judge-results/`. Ou seja, esta parte repete exatamente o pós-processamento do artigo, sem rodar inferência de novo — por isso não há de onde vir variação. Isso mudaria se, na Parte 5, você enviasse as mesmas perguntas de novo a um modelo via API: aí entrariam em jogo não-determinismo de amostragem (temperatura > 0), possíveis atualizações silenciosas do modelo por trás de um alias do OpenRouter (ex.: "gpt-5" hoje pode não ser byte-a-byte o mesmo checkpoint usado no artigo) e até variação do próprio juiz LLM. A Parte 3.2 (variabilidade entre `first_run`/`second_run`) existe justamente para dar uma ideia de quanta variação uma segunda rodada real introduz — isso é uma adição deste kit de reprodução, **não está no artigo original**, que reporta apenas uma rodada por configuração.

**O que os números confirmam (e aprofundam) das conclusões do artigo:**

- **Modelos de fronteira dominam, mas não igualmente.** GPT-5 lidera quase todas as configurações (96–100% em MC nos níveis 1–4 de pt-PT), com Qwen3-235B-A22B — o maior modelo *open-weight* testado — logo atrás, e por vezes à frente em pt-BR (ex.: nível 3, OE: 93,10% vs. 91,38% do GPT-5). Isso confirma a leitura do artigo de que o Qwen3-235B "exibe desempenho equiparável aos modelos fechados".
- **A lacuna MC vs. OE é real e cresce com a dificuldade.** Para quase todo modelo, acurácia em múltipla escolha > questões abertas no mesmo nível — esperado, já que MC dá ~20% de chance de acerto ao acaso (5 opções) e não exige produzir a resposta do zero. A lacuna é pequena para o GPT-5 (pt-PT nível 4: 96,55% MC vs. 85,52% OE, ~11 pp) e maior para modelos mais fracos em contextos favoráveis (Gemma-3-27b-it, pt-BR nível 4: 64,81% MC vs. 52,63% OE, ~12 pp) — mas em modelos muito fracos a lacuna às vezes encolhe porque o modelo já erra muito nos dois formatos (Llama-3.3-70B, pt-PT nível 3: 23,78% MC vs. 18,13% OE).
- **Figuras são o maior fator de degradação em pt-PT.** Comparando `sem figura` vs. `com figura` na Tabela 3, a queda é sistemática e às vezes brutal: Llama-3.3-70B cai de 32,29% (sem figura) para 6,38% (com figura) em MC no nível 3 — quase 26 pp — porque o modelo recebe apenas o código LaTeX da figura, não a imagem renderizada, e precisa inferir a geometria a partir do código-fonte. Mesmo o GPT-5, o mais robusto, cai 9–13 pp em quase todos os níveis quando há figura. Isso reforça o achado central do artigo: representar figuras como texto (LaTeX) ainda deixa um "buraco" de raciocínio visual que nem os melhores modelos fecham totalmente.
- **Escala ajuda, mas com retornos decrescentes e nem sempre monótonos.** Na família Qwen3 (Tabela 4), 8B → 14B traz o salto mais visível (pt-PT nível 3, OE: 61,14% → 76,68%), mas 14B → 32B → 235B nem sempre melhora em todas as configurações — o modelo *mixture-of-experts* 30B-A3B (poucos parâmetros ativos por token) frequentemente fica abaixo do denso 14B, sugerindo que ativação total conta mais do que contagem nominal de parâmetros para raciocínio matemático.
- **Cuidado ao comparar pt-PT com pt-BR diretamente.** Os dois idiomas não cobrem os mesmos níveis (pt-PT: 1–4, com figuras; pt-BR: 2–5, sem nenhuma figura) nem vêm das mesmas competições — são amostras de tamanhos e fontes diferentes. O nível 1 de pt-PT, por exemplo, tem só 78 questões de MC, o que torna a acurácia mais ruidosa ponto a ponto do que níveis com centenas de questões, como o nível 4 de pt-BR (270). Uma acurácia mais alta em pt-BR num nível pode refletir a ausência de figuras ali, não necessariamente o modelo "ser melhor em português brasileiro".

Em resumo: este notebook não só repete os números do artigo — ele mostra, de forma auditável e determinística, **de onde** cada número da Tabela 2/3/4 vem, e reforça com evidência quantitativa exata os três eixos de dificuldade que o MATH-PT foi desenhado para expor: tipo de questão (MC vs. OE), presença de figura e nível de dificuldade.

## Parte 5 — Gerar novos dados do zero (opcional)

Esta parte reproduz o pipeline de geração original (`create_prompts.ipynb` → envio ao modelo → `create_judge_prompts.ipynb` → envio ao juiz → avaliação), a partir dos dados atuais em `data/*.json`, para testar **novos modelos** que ainda não têm resultados publicados no repositório. **Requer `OPEN_ROUTER_API_KEY`** (Parte 1) ou um servidor local compatível com a API da OpenAI (ex.: vLLM).

Os arquivos gerados aqui usam o sufixo `-colab` para não sobrescrever os arquivos originais usados na Parte 3.

> Nota sobre `id`: os arquivos `data/*.json` (nova estrutura do dataset) não trazem mais o campo `id` original de cada questão — apenas `problem`/`choices`/`answer`/`solution`/`level`. Por isso, os `id`s usados a partir daqui são reatribuídos sequencialmente (posição no arquivo), válidos apenas dentro desta geração — não os compare com os `id`s dos arquivos pré-existentes em `prompts/`/`results/`.

### 5.1 — Carregar pares pergunta-resposta

Diferente do `create_prompts.ipynb` original, não é necessário expandir referências a imagens/tabelas aqui: os arquivos `data/*.json` (nova estrutura, ver `data/old_structure/convert-data.ipynb`) já trazem esse conteúdo expandido diretamente no texto de `problem`/`solution`.

In [ ]:
question_type = "open-ended"  # "multiple-choice" ou "open-ended"
dataset = "ptbr"  # "ptbr" ou "ptpt"
prompt_language = dataset  # "en", "ptbr" ou "ptpt"

_file_question_type = "multiple_choice" if question_type == "multiple-choice" else "open_ended"
data_path = f"data/{dataset}_{_file_question_type}_test.json"

with open(data_path, "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

print(f"Carregados {len(qa_pairs)} pares pergunta-resposta de {data_path}")


### 5.2 — Construir prompts

In [ ]:
def build_prompt(question_type, question_text, options_text):
    if question_type == "multiple-choice":
        ordered_keys = [k for k in ["A", "B", "C", "D", "E"] if k in options_text]
        options_lines = "\n".join(f"{k}) {options_text[k]}" for k in ordered_keys)

        if prompt_language == "en":
            header = (
                "Solve the following math multiple-choice question. Make sure to put the correct "
                "option letter (A,B,C,D or E), and only the correct option letter, inside \\boxed{}.\n\n"
            )
            prompt = f"{header}Question: {question_text}\n\n{options_lines}\n"
        elif prompt_language == "ptbr":
            header = (
                "Resolva a seguinte questão de múltipla escolha de matemática. Certifique-se de colocar "
                "a letra da opção correta (A,B,C,D or E), e somente a letra da opção correta, dentro de "
                "\\boxed{}. Use Português do Brasil para pensar e responder.\n\n"
            )
            prompt = f"{header}Questão: {question_text}\n\n{options_lines}\n"
        elif prompt_language == "ptpt":
            header = (
                "Resolve a seguinte questão de escolha múltipla de matemática. Certifica-te de colocar "
                "a letra da opção correta (A,B,C,D ou E), e apenas a letra da opção correta, dentro de "
                "\\boxed{}. Usa Português Europeu para pensar e responder.\n\n"
            )
            prompt = f"{header}Questão: {question_text}\n\n{options_lines}\n"
        question_only = f"{question_text}\n\n{options_lines}\n"
    elif question_type == "open-ended":
        if prompt_language == "en":
            header = "Solve the following math open-ended question. Make sure to put the final answer inside \\boxed{}.\n\n"
            prompt = f"{header}Question: {question_text}\n"
        elif prompt_language == "ptbr":
            header = (
                "Resolva a seguinte questão aberta de matemática. Certifique-se de colocar a resposta "
                "final dentro de \\boxed{}. Use Português do Brasil para pensar e responder.\n\n"
            )
            prompt = f"{header}Questão: {question_text}\n"
        elif prompt_language == "ptpt":
            header = (
                "Resolve a seguinte questão de resposta aberta de matemática. Certifica-te de colocar a "
                "resposta final dentro de \\boxed{}. Usa Português Europeu para pensar e responder.\n\n"
            )
            prompt = f"{header}Questão: {question_text}\n"
        question_only = f"{question_text}\n"
    return question_only, prompt


prompts_new = []
for idx, item in enumerate(qa_pairs):
    question_text = item.get("problem")
    if question_type == "multiple-choice":
        options = item.get("choices") or {}
        correct_answer = item.get("answer")
        if not question_text or not options:
            print(f"Pulando item sem pergunta/opções válidas (índice {idx})")
            continue
    else:
        options = {}
        correct_answer = item.get("solution")
        if not question_text:
            print(f"Pulando item sem pergunta válida (índice {idx})")
            continue

    question_only, prompt_text = build_prompt(question_type, question_text, options)
    prompts_new.append({
        "id": idx,
        "question": question_only,
        "correct_answer": correct_answer,
        "level": item.get("level"),
        "contains_latex_figure_in_question": bool(item.get("question_has_figure", False)),
        "prompt": prompt_text,
    })

os.makedirs("prompts", exist_ok=True)
prompts_new_path = f"prompts/{dataset}-{question_type}-prompts-prompt-language-{prompt_language}-colab.json"
with open(prompts_new_path, "w", encoding="utf-8") as f:
    json.dump(prompts_new, f, ensure_ascii=False, indent=2)

print(f"{len(prompts_new)} prompts construídos e salvos em {prompts_new_path}")


### 5.3 — Enviar prompts a um novo modelo

Use `base_url`/`api_key` do OpenRouter, ou de um servidor local (ex.: vLLM `http://localhost:8000/v1`, exposto via túnel se o servidor não roda no próprio Colab).

In [ ]:
new_model = "qwen/qwen3-14b"  # nome do modelo no OpenRouter, ou caminho/nome do modelo em um servidor vLLM
new_model_tag = "qwen3-14b-colab"  # usado apenas no nome dos arquivos de saída
base_url = "https://openrouter.ai/api/v1"  # ou "http://localhost:8000/v1" para vLLM local
api_key = "openrouter"  # ou uma chave literal (ex.: "token-abc123") para um servidor local
concurrency = 8

responses_new_path = f"results/{question_type}/{new_model_tag}-{dataset}-responses-prompt-language-{prompt_language}-colab.json"

await send_prompts_to_model(
    prompts_path=prompts_new_path,
    responses_path=responses_new_path,
    model=new_model,
    base_url=base_url,
    api_key=api_key,
    concurrency=concurrency,
)


### 5.4 — Carregar respostas e avaliar (se múltipla escolha)

In [ ]:
new_responses = {new_model_tag: []}
with open(responses_new_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            new_responses[new_model_tag].append(json.loads(line))

prompts_new_by_id = {p["id"]: p for p in prompts_new}

if question_type == "multiple-choice":
    acc_all, acc_with_fig, acc_no_fig = compute_mc_accuracy({new_model_tag}, new_responses, prompts_new_by_id)
    print(format_acc_report(f"{new_model_tag} — Overall", acc_all, {new_model_tag}))
else:
    print("question_type != 'multiple-choice' — pulando avaliação de múltipla escolha (veja 4.5-4.7 para questões abertas).")


### 5.5 — Construir prompts para o juiz (questões abertas)

Equivalente a `create_judge_prompts.ipynb`. Só se aplica quando `question_type == "open-ended"`.

In [ ]:
def build_judge_prompt(model_answer, golden_answer):
    if not model_answer or model_answer.strip() == "":
        model_answer = "I could not provide an answer. Consider it as incorrect."
    return f"""
You are an impartial judge. Your task is to evaluate the correctness of a model's
answer to an open-ended math question by comparing it ONLY to the gold solution.

The model's answer may use different reasoning steps or formatting than the
gold solution; this is acceptable **as long as the final mathematical result
is equivalent**. Focus strictly on mathematical correctness.

Instructions:
1. Compare the **final result** of the model answer to the gold answer.
2. Minor algebraic, arithmetic, or formatting differences are allowed.
3. Completely ignore writing style, explanation detail, or formatting.
4. If the final results match or are mathematically equivalent → score 1.
5. If the final results do not match → score 0.
6. Output ONLY a JSON object in this exact format:

{{
  "score": 0 or 1,
  "explanation": "short explanation of the comparison"
}}

[BEGIN DATA]
************
[Golden Answer]: {golden_answer}
************
[Model Answer]: {model_answer}
************
[END DATA]

Provide your judgment now.
"""


judge_prompts_new_path = None
if question_type == "open-ended":
    judge_prompts_new = []
    for item in new_responses[new_model_tag]:
        model_answer = _extract_content(item)
        golden_item = prompts_new_by_id.get(item.get("id"))
        if golden_item is None:
            continue
        judge_prompts_new.append({
            "id": item.get("id"),
            "level": golden_item.get("level"),
            "contains_latex_figure_in_question": golden_item.get("contains_latex_figure_in_question"),
            "prompt": build_judge_prompt(model_answer, golden_item.get("correct_answer")),
        })

    os.makedirs("judge-prompts", exist_ok=True)
    judge_prompts_new_path = f"judge-prompts/{new_model_tag}-{dataset}-judge-prompts-prompt-language-{prompt_language}-colab.json"
    with open(judge_prompts_new_path, "w", encoding="utf-8") as f:
        json.dump(judge_prompts_new, f, ensure_ascii=False, indent=2)
    print(f"{len(judge_prompts_new)} judge prompts salvos em {judge_prompts_new_path}")
else:
    print("question_type != 'open-ended' — pulando esta célula.")


### 5.6 — Enviar prompts do juiz

In [ ]:
if question_type == "open-ended":
    judge_model = "moonshotai/kimi-k2-thinking"  # modelo juiz no OpenRouter
    judge_tag = "kimi-k2-thinking"

    judge_responses_new_path = (
        f"judge-results/{new_model_tag}-{dataset}-judge-responses-"
        f"prompt-language-{prompt_language}-judge-{judge_tag}-colab.json"
    )

    await send_prompts_to_model(
        prompts_path=judge_prompts_new_path,
        responses_path=judge_responses_new_path,
        model=judge_model,
        base_url="https://openrouter.ai/api/v1",
        api_key="openrouter",
        concurrency=concurrency,
    )
else:
    print("question_type != 'open-ended' — pulando esta célula.")


### 5.7 — Avaliar respostas abertas (novo modelo)

In [ ]:
if question_type == "open-ended":
    new_judge_responses = {new_model_tag: []}
    with open(judge_responses_new_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                new_judge_responses[new_model_tag].append(json.loads(line))

    for response in new_judge_responses[new_model_tag]:
        text = _extract_content(response)
        m_score = score_re.search(text) if text else None
        m_expl = explanation_re.search(text) if text else None
        response["score"] = int(m_score.group(1)) if m_score else None
        response["explanation"] = m_expl.group(1) if m_expl else None

    judge_prompts_by_model_new = {new_model_tag: {p["id"]: p for p in judge_prompts_new}}
    acc_all, acc_with_fig, acc_no_fig = compute_oe_accuracy({new_model_tag}, new_judge_responses, judge_prompts_by_model_new)
    print(format_acc_report(f"{new_model_tag} — Overall", acc_all, {new_model_tag}))
else:
    print("question_type != 'open-ended' — pulando esta célula.")


## Parte 6 — Exportar resultados

In [ ]:
import shutil
from datetime import datetime

zip_name = f"math-benchmark-resultados-{datetime.now():%Y%m%d-%H%M%S}"
shutil.make_archive(zip_name, "zip", root_dir=".", base_dir="results")

try:
    from google.colab import files
    files.download(f"{zip_name}.zip")
except ImportError:
    print(f"Fora do Colab — arquivo salvo em: {zip_name}.zip")


In [1]:
# Alternativa: copiar para o Google Drive (requer ter montado o Drive na Parte 1)
import shutil
shutil.copytree("results", "/content/drive/MyDrive/math-benchmark-results", dirs_exist_ok=True)
print("Resultados copiados para o Google Drive.")


Resultados copiados para o Google Drive.
